# Week 01 - Advanced RAG with Services

**Objective**: Build RAG, CAG, and CRAG using service layer.

**Architecture**: Uses `RAGService`, `CAGService`, `CRAGService`, and `CAGCache` from `application.chat_service`

**Provider Support**: Uses OpenRouter unified API for multi-provider LLM access (GPT-4o, Claude, Gemini, etc.) or direct OpenAI


In [1]:
# Cell 1: Setup & Installations
import sys

print("[SETUP] Checking required packages...")
required = [
    "langchain",
    "langchain_core",
    "langchain_openai",
    "langchain_community",
    "chromadb",
    "python-dotenv",
    "pandas"
]

missing = []
for pkg in required:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"[INSTALL] Installing missing packages: {missing}")
    %pip install -q {" ".join(missing)}

print("[SUCCESS] Packages ready")


[SETUP] Checking required packages...


C:\Users\naduk\AppData\Local\Temp\ipykernel_18964\1374950540.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  __import__(pkg.replace("-", "_"))


[INSTALL] Installing missing packages: ['python-dotenv']
Note: you may need to restart the kernel to use updated packages.
[SUCCESS] Packages ready


d:\portfolio projects\hospital-system\.venv\Scripts\python.exe: No module named pip


In [2]:
# Cell 2: Imports & Environment Setup
import os
import sys
import time
from pathlib import Path
from dotenv import load_dotenv

# Add project root src to path
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if (notebook_dir.parent / "src").exists() else notebook_dir
sys.path.insert(0, str(project_root / "src"))

# Load environment
load_dotenv(project_root / ".env")

# Check for API key (OpenRouter preferred, OpenAI as fallback)
openrouter_key = os.getenv("OPENROUTER_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

if not openrouter_key and not openai_key:
    raise EnvironmentError(
        "[ERROR] No API key found!\n"
        "   Add OPENROUTER_API_KEY (recommended) or OPENAI_API_KEY to .env"
    )

# Load configuration
from utils.config_utils import (
    VECTOR_DIR, CACHE_DIR, TOP_K_RESULTS,
    CHAT_MODEL, EMBEDDING_MODEL, PROVIDER
)

provider = "OpenRouter" if openrouter_key else "OpenAI"
print("[SUCCESS] Environment loaded")
print(f"Provider: {provider}")
print(f"Project root: {project_root}")


[SUCCESS] Environment loaded
Provider: OpenRouter
Project root: d:\portfolio projects\hospital-system


## Import Chat Services

Using RAG/CAG/CRAG services from application layer:
- `RAGService`: Standard RAG with modern LCEL
- `CAGService`: Cache-Augmented Generation (semantic)
- `CRAGService`: Corrective RAG with confidence scoring
- `CAGCache`: Semantic cache (FAQs + History)


In [3]:
# Cell 3: Import Chat Services
from application.chat_service import (
    RAGService,
    CAGService,
    CRAGService,
    CAGCache
)

print("[SUCCESS] Chat services loaded from service layer")
print("Location: application.chat_service")
print("\nAvailable services:")
print("   1. RAGService   - Standard RAG with modern LCEL")
print("   2. CAGService   - Cache-Augmented Generation (semantic)")
print("   3. CRAGService  - Corrective RAG with confidence scoring")
print("   4. CAGCache     - Semantic cache (FAQs + History)")


d:\portfolio projects\hospital-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[SUCCESS] Chat services loaded from service layer
Location: application.chat_service

Available services:
   1. RAGService   - Standard RAG with modern LCEL
   2. CAGService   - Cache-Augmented Generation (semantic)
   3. CRAGService  - Corrective RAG with confidence scoring
   4. CAGCache     - Semantic cache (FAQs + History)


In [4]:
# Cell 4: Connect to Vector Store & Initialize LLM
from langchain_community.vectorstores import Chroma
from infrastructure.llm_providers import (
    get_default_embeddings,
    get_chat_llm
)

# Initialize embeddings (using local embeddings for zero cost, or 'openrouter')
embeddings = get_default_embeddings(provider="local")
llm = get_chat_llm(temperature=0.0, max_tokens=500)

print(f"[INFO] LLM initialized: {CHAT_MODEL}")
print(f"[INFO] Embeddings initialized: {EMBEDDING_MODEL} (local)")
print(f"[INFO] Provider: {PROVIDER}")

# Connect to vector store
if not VECTOR_DIR.exists():
    raise FileNotFoundError(f"[ERROR] Vector store directory not found at {VECTOR_DIR}. Run 02_chunk_and_embeded.ipynb first.")

vectorstore = Chroma(
    persist_directory=str(VECTOR_DIR),
    embedding_function=embeddings,
    collection_name="nawaloka"
)

# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K_RESULTS}
)

print("[SUCCESS] Connected to vector store")
print(f"Collection size: {vectorstore._collection.count()} vectors")


2026-11-06 19:33:18,000 - infrastructure.llm_providers.embeddings - INFO - Using local HuggingFace embeddings: model=all-MiniLM-L6-v2
d:\portfolio projects\hospital-system\src\infrastructure\llm_providers\embeddings.py:61: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(model_name=model_name)
2026-11-06 19:33:18,005 - sentence_transformers.base.model - INFO - No device provided, using cpu


D:\portfolio projects\hospital-system


2026-11-06 19:33:18,465 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-11-06 19:33:18,515 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
2026-11-06 19:33:18,794 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-11-06 19:33:18,852 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-11-06 19:33:18,852 - sentence_transformers.base.model - INFO - Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
2026-11-0

[INFO] LLM initialized: openai/gpt-4o-mini
[INFO] Embeddings initialized: openai/text-embedding-3-large (local)
[INFO] Provider: openrouter
[SUCCESS] Connected to vector store
Collection size: 50 vectors


C:\Users\naduk\AppData\Local\Temp\ipykernel_18964\3301709257.py:20: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


---
# 1. Standard RAG with Service


In [5]:
# Cell 5: Initialize RAG Service
rag_service = RAGService(
    retriever=retriever,
    llm=llm,
    k=TOP_K_RESULTS
)

print("[SUCCESS] RAGService initialized")
print(f"Retrieval target: top-{TOP_K_RESULTS} documents")


2026-11-06 19:33:26,121 - application.chat_service.rag_service - INFO - Initializing RAGService instance with top_k=4
2026-11-06 19:33:26,124 - application.chat_service.rag_service - INFO - Constructing LCEL RAG Chain...
2026-11-06 19:33:26,126 - application.chat_service.rag_service - INFO - LCEL RAG Chain constructed successfully.


[SUCCESS] RAGService initialized
Retrieval target: top-4 documents


In [6]:
# Cell 6: Generate Answer with RAG Service
USER_QUERY = "Tell me about Nawaloka's cardiology services and health check packages."

print(f"Query: {USER_QUERY}\n")
print("=" * 80)
print("GENERATING ANSWER WITH RAG SERVICE...")
print("=" * 80)

result = rag_service.generate(USER_QUERY)

print(f"\nGeneration time: {result['generation_time']:.2f}s")
print(f"Documents used: {result['num_docs']}")
print("\n" + "=" * 80)
print("ANSWER")
print("=" * 80)
print(result['answer'])
print("\n" + "=" * 80)
print("EVIDENCE URLS")
print("=" * 80)
for url in result['evidence_urls']:
    print(f"  - {url}")


2026-11-06 19:33:26,153 - application.chat_service.rag_service - INFO - Processing generation request for query: 'Tell me about Nawaloka's cardiology services and health check packages.'


Query: Tell me about Nawaloka's cardiology services and health check packages.

GENERATING ANSWER WITH RAG SERVICE...


2026-11-06 19:33:26,214 - application.chat_service.rag_service - INFO - Retrieved 4 evidence documents.
2026-11-06 19:33:26,242 - utils.format_utils - INFO - Formatting 4 documents into context.
2026-11-06 19:33:27,440 - httpx2 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-06 19:33:29,877 - application.chat_service.rag_service - INFO - Generation complete in 3.72s | Docs: 4 | Unique URLs: 2



Generation time: 3.72s
Documents used: 4

ANSWER
1. **Key Facts**:
   - Nawaloka Hospital offers a range of medical specialties, including cardiology.
   - The hospital is recognized for advanced medical treatment and trusted specialists.
   - Patients can easily book appointments with experienced consultants through their channeling services.

2. **Answer**: While specific details about cardiology services and health check packages are not provided in the available context, Nawaloka Hospital does have a cardiology department where patients can consult experienced specialists. For more information on health check packages, please contact the hospital directly [URL]. 

3. **Contact**: For specific questions regarding cardiology services and health check packages, please call +94 11 544 4444.

EVIDENCE URLS
  - https://www.nawaloka.com/
  - https://www.nawaloka.com/channeling


---
# 2. Cache-Augmented Generation (CAG) with Semantic Matching

CAG uses lightweight semantic similarity to cache responses:
- FAQs: Static questions, never expire
- History: User queries, 24-hour TTL
- Matching: Cosine similarity (catches paraphrased queries)


In [7]:
# Cell 7: Initialize CAG Service with Semantic Cache
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Create semantic cache (embedder required for similarity matching)
cache = CAGCache(
    cache_dir=CACHE_DIR,
    embedder=embeddings,  # Uses same embeddings as vector store
    similarity_threshold=0.90,  # Catches paraphrased questions
    history_ttl_hours=24.0  # History expires after 24 hours
)

# Create CAG service
cag_service = CAGService(
    rag_service=rag_service,
    cache=cache
)

print("[SUCCESS] CAGService initialized (semantic matching)")
print(f"Cache directory: {CACHE_DIR}")
stats = cache.stats()
print(f"Cached responses: {stats['total_cached']}")
print(f"Similarity threshold: {stats['similarity_threshold']}")
print(f"History TTL: {stats['history_ttl_hours']}h")


2026-11-06 19:33:29,895 - application.chat_service.cag_cache - WARNING - No valid FAQs with responses found. Resetting FAQ matrix to empty.


[SUCCESS] CAGService initialized (semantic matching)
Cache directory: D:\portfolio projects\hospital-system\data\cag_cache
Cached responses: 0
Similarity threshold: 0.9
History TTL: 24.0h


### Load Known FAQs

FAQs are defined in `config/faq.yaml` and can be:
1. Loaded into cache (registers questions)
2. Warmed (generates responses via RAG)

Once warmed, FAQs provide instant responses for common questions.


In [8]:
# Cell 8: Load Known FAQs from Config
from utils.config_utils import KNOWN_FAQS

print(f"[INFO] Found {len(KNOWN_FAQS)} FAQs in config/faq.yaml")
print("\nSample FAQs:")
for faq in KNOWN_FAQS[:5]:
    print(f"   - {faq}")
if len(KNOWN_FAQS) > 5:
    print(f"   ... and {len(KNOWN_FAQS) - 5} more\n")

# Load FAQs into cache (registers questions)
loaded = cag_service.load_faqs(KNOWN_FAQS)
print(f"[SUCCESS] Loaded {loaded} new FAQs into cache")

# To warm FAQs (pre-generate responses), uncomment:
# print("\nWarming FAQs...")
# cag_service.warm_faqs()

print("\nTip: Run cag_service.warm_faqs() to pre-generate FAQ responses")


[INFO] Found 22 FAQs in config/faq.yaml

Sample FAQs:
   - What are the visiting hours at Nawaloka Hospital?
   - What are the opening hours of Nawaloka Hospital?
   - Where is Nawaloka Hospital located?
   - What is the address of Nawaloka Hospital?
   - How do I contact Nawaloka Hospital?
   ... and 17 more



2026-11-06 19:33:30,244 - application.chat_service.cag_cache - INFO - Saved 22 items to FAQ cache on disk.
2026-11-06 19:33:30,244 - application.chat_service.cag_cache - WARNING - No valid FAQs with responses found. Resetting FAQ matrix to empty.
2026-11-06 19:33:30,244 - application.chat_service.cag_cache - INFO - Loaded 22 new FAQ(s). Total FAQs now: 22


[SUCCESS] Loaded 22 new FAQs into cache

Tip: Run cag_service.warm_faqs() to pre-generate FAQ responses


In [9]:
# Cell 9: Test CAG Performance
print("=" * 80)
print("CAG PERFORMANCE TEST")
print("=" * 80)

test_queries = [
    "What are the visiting hours at Nawaloka?",
    "How do I contact Nawaloka Hospital?",
    "What services does Nawaloka provide?"
]

# First run: populate cache
print("\n[STEP 1] FIRST RUN (Populating cache)...\n")
for query in test_queries:
    result = cag_service.generate(query, use_cache=True, verbose=False)
    print(f"   Query: {query[:50]}...")
    print(f"   Time: {result['generation_time']:.2f}s | Cache hit: {result['cache_hit']}")
    print()

# Second run: cache hits
print("\n[STEP 2] SECOND RUN (Using cache)...\n")
for query in test_queries:
    result = cag_service.generate(query, use_cache=True, verbose=False)
    print(f"   Query: {query[:50]}...")
    print(f"   Time: {result['generation_time']:.2f}s | Cache hit: {result['cache_hit']}")
    if result['cache_hit']:
        print("   [CACHE HIT] Instant response from cache!")
    print()

print("\nCache Statistics:")
stats = cache.stats()
print(f"   Total cached: {stats['total_cached']}")
print(f"   Cache size: {stats['cache_size_kb']:.2f} KB")


2026-11-06 19:33:30,262 - application.chat_service.cag_cache - INFO - Initiating cache lookup for query: 'What are the visiting hours at Nawaloka?'
2026-11-06 19:33:30,277 - application.chat_service.cag_cache - INFO - MISS [All Tiers] | No matching cached entries found for query: 'What are the visiting hours at Nawaloka?'
2026-11-06 19:33:30,277 - application.chat_service.rag_service - INFO - Processing generation request for query: 'What are the visiting hours at Nawaloka?'
2026-11-06 19:33:30,293 - application.chat_service.rag_service - INFO - Retrieved 4 evidence documents.
2026-11-06 19:33:30,321 - utils.format_utils - INFO - Formatting 4 documents into context.


CAG PERFORMANCE TEST

[STEP 1] FIRST RUN (Populating cache)...



2026-11-06 19:33:30,988 - httpx2 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-06 19:33:33,121 - application.chat_service.rag_service - INFO - Generation complete in 2.84s | Docs: 4 | Unique URLs: 2
2026-11-06 19:33:33,138 - application.chat_service.cag_cache - INFO - Cached new query to history. Generated key: 'e2051b28de69da25aab8d97a69b24e9e'
2026-11-06 19:33:33,138 - application.chat_service.cag_cache - INFO - Updated History embedding matrix: 1 active entries | Dimensions: 384
2026-11-06 19:33:33,143 - application.chat_service.cag_cache - INFO - Saved 1 items to History cache on disk.
2026-11-06 19:33:33,144 - application.chat_service.cag_cache - INFO - Initiating cache lookup for query: 'How do I contact Nawaloka Hospital?'
2026-11-06 19:33:33,163 - application.chat_service.cag_cache - INFO - Updated History embedding matrix: 1 active entries | Dimensions: 384
2026-11-06 19:33:33,164 - application.chat_service.cag_cache - INFO

   Query: What are the visiting hours at Nawaloka?...
   Time: 2.84s | Cache hit: False



2026-11-06 19:33:33,917 - httpx2 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-06 19:33:36,352 - application.chat_service.rag_service - INFO - Generation complete in 3.19s | Docs: 4 | Unique URLs: 2
2026-11-06 19:33:36,370 - application.chat_service.cag_cache - INFO - Cached new query to history. Generated key: 'f42e25fd22c0cc6d6da5e9462f5fdd27'
2026-11-06 19:33:36,370 - application.chat_service.cag_cache - INFO - Updated History embedding matrix: 2 active entries | Dimensions: 384
2026-11-06 19:33:36,370 - application.chat_service.cag_cache - INFO - Saved 2 items to History cache on disk.
2026-11-06 19:33:36,370 - application.chat_service.cag_cache - INFO - Initiating cache lookup for query: 'What services does Nawaloka provide?'
2026-11-06 19:33:36,387 - application.chat_service.cag_cache - INFO - Updated History embedding matrix: 2 active entries | Dimensions: 384
2026-11-06 19:33:36,387 - application.chat_service.cag_cache - INF

   Query: How do I contact Nawaloka Hospital?...
   Time: 3.19s | Cache hit: False



2026-11-06 19:33:37,121 - httpx2 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-06 19:33:39,835 - application.chat_service.rag_service - INFO - Generation complete in 3.45s | Docs: 4 | Unique URLs: 2
2026-11-06 19:33:39,852 - application.chat_service.cag_cache - INFO - Cached new query to history. Generated key: '9f1ded7aa0176a697f5bfb9ae131bc9f'
2026-11-06 19:33:39,854 - application.chat_service.cag_cache - INFO - Updated History embedding matrix: 3 active entries | Dimensions: 384
2026-11-06 19:33:39,855 - application.chat_service.cag_cache - INFO - Saved 3 items to History cache on disk.
2026-11-06 19:33:39,855 - application.chat_service.cag_cache - INFO - Initiating cache lookup for query: 'What are the visiting hours at Nawaloka?'
2026-11-06 19:33:39,877 - application.chat_service.cag_cache - INFO - Updated History embedding matrix: 3 active entries | Dimensions: 384
2026-11-06 19:33:39,879 - application.chat_service.cag_cache -

   Query: What services does Nawaloka provide?...
   Time: 3.45s | Cache hit: False


[STEP 2] SECOND RUN (Using cache)...

   Query: What are the visiting hours at Nawaloka?...
   Time: 0.02s | Cache hit: True
   [CACHE HIT] Instant response from cache!

   Query: How do I contact Nawaloka Hospital?...
   Time: 0.02s | Cache hit: True
   [CACHE HIT] Instant response from cache!

   Query: What services does Nawaloka provide?...
   Time: 0.03s | Cache hit: True
   [CACHE HIT] Instant response from cache!


Cache Statistics:
   Total cached: 25
   Cache size: 80.81 KB


---
# 3. Corrective RAG (CRAG)


In [10]:
# Cell 10: Initialize CRAG Service
from utils.config_utils import CRAG_EXPANDED_K

crag_service = CRAGService(
    retriever=retriever,
    llm=llm,
    initial_k=TOP_K_RESULTS,
    expanded_k=CRAG_EXPANDED_K
)

print("[SUCCESS] CRAGService initialized")
print(f"Initial retrieval target: top-{TOP_K_RESULTS}")
print(f"Corrective retrieval target: top-{CRAG_EXPANDED_K}")


2026-11-06 19:33:39,936 - application.chat_service.crag_service - INFO - Initialized CRAGService with initial_k=4 and expanded_k=8


[SUCCESS] CRAGService initialized
Initial retrieval target: top-4
Corrective retrieval target: top-8


# Test CRAG with Different Query Types


In [11]:
# Cell 11: Test CRAG with Different Query Types
print("=" * 80)
print("CORRECTIVE RAG (CRAG) TEST")
print("=" * 80)

test_cases = [
    {
        'query': "cardiology",
        'label': "Vague query (evaluates confidence and applies correction)"
    },
    {
        'query': "What are Nawaloka Hospital's cardiology services and facilities?",
        'label': "Specific query (sufficient confidence)"
    }
]

for i, test in enumerate(test_cases, 1):
    print(f"\nTest {i}/{len(test_cases)}: {test['label']}")
    print("-" * 80)
    
    result = crag_service.generate(test['query'], confidence_threshold=0.6, verbose=False)
    
    print(f"\nResult:")
    print(f"   Initial confidence: {result['confidence_initial']:.2f}")
    print(f"   Final confidence: {result['confidence_final']:.2f}")
    print(f"   Correction applied: {result['correction_applied']}")
    print(f"   Documents used: {result['docs_used']}")
    print(f"   Generation time: {result['generation_time']:.2f}s")
    print("\n" + "-" * 80)


2026-11-06 19:33:39,985 - utils.format_utils - INFO - Confidence Breakdown -> Keyword: 0.500 | Length: 0.431 | Diversity: 0.333 | Final: 0.446
2026-11-06 19:33:40,005 - utils.format_utils - INFO - Confidence Breakdown -> Keyword: 0.250 | Length: 0.609 | Diversity: 0.333 | Final: 0.374
2026-11-06 19:33:40,005 - utils.format_utils - INFO - Formatting 8 documents into context.


CORRECTIVE RAG (CRAG) TEST

Test 1/2: Vague query (evaluates confidence and applies correction)
--------------------------------------------------------------------------------


2026-11-06 19:33:40,652 - httpx2 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-06 19:33:42,823 - utils.format_utils - INFO - Confidence Breakdown -> Keyword: 0.406 | Length: 1.000 | Diversity: 0.333 | Final: 0.570
2026-11-06 19:33:42,852 - utils.format_utils - INFO - Confidence Breakdown -> Keyword: 0.297 | Length: 1.000 | Diversity: 0.333 | Final: 0.515
2026-11-06 19:33:42,852 - utils.format_utils - INFO - Formatting 8 documents into context.



Result:
   Initial confidence: 0.45
   Final confidence: 0.37
   Correction applied: True
   Documents used: 8
   Generation time: 2.79s

--------------------------------------------------------------------------------

Test 2/2: Specific query (sufficient confidence)
--------------------------------------------------------------------------------


2026-11-06 19:33:43,468 - httpx2 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"



Result:
   Initial confidence: 0.57
   Final confidence: 0.52
   Correction applied: True
   Documents used: 8
   Generation time: 3.39s

--------------------------------------------------------------------------------


In [12]:
# Cell 12: Check All Services Ready
print("Checking if all services are initialized...\n")

services_ready = True

# Check RAGService
try:
    rag_service
    print("[READY] RAGService: Ready")
except NameError:
    print("[MISSING] RAGService: NOT initialized")
    services_ready = False

# Check CAGService
try:
    cag_service
    print("[READY] CAGService: Ready")
except NameError:
    print("[MISSING] CAGService: NOT initialized")
    services_ready = False

# Check CRAGService
try:
    crag_service
    print("[READY] CRAGService: Ready")
except NameError:
    print("[MISSING] CRAGService: NOT initialized")
    services_ready = False

# Check Vector Store
try:
    vectorstore
    print("[READY] Vector Store: Connected")
except NameError:
    print("[MISSING] Vector Store: NOT connected")
    services_ready = False

print("\n" + "=" * 80)
if services_ready:
    print("[SUCCESS] All services ready! You can run the remaining cells.")
else:
    print("[WARNING] Some services are missing. Please run all cells above first.")
print("=" * 80)


Checking if all services are initialized...

[READY] RAGService: Ready
[READY] CAGService: Ready
[READY] CRAGService: Ready
[READY] Vector Store: Connected

[SUCCESS] All services ready! You can run the remaining cells.


---
# Comprehensive Comparison: RAG vs CAG vs CRAG


In [13]:
# Cell 13: Comprehensive Comparison
import pandas as pd

print("=" * 80)
print("RAG vs CAG vs CRAG COMPARISON")
print("=" * 80)

comparison_query = "What cardiology services are available at Nawaloka?"

# Test 1: Standard RAG
print(f"\n1. Standard RAG...")
rag_result = rag_service.generate(comparison_query)
print(f"   Time: {rag_result['generation_time']:.2f}s")

# Test 2: CAG (uses cache on repeated/similar queries)
print(f"\n2. Cache-Augmented Generation (CAG)...")
cag_result = cag_service.generate(comparison_query, use_cache=True, verbose=False)
print(f"   Time: {cag_result['generation_time']:.2f}s")
print(f"   Cache hit: {cag_result['cache_hit']}")

# Test 3: CRAG
print(f"\n3. Corrective RAG (CRAG)...")
crag_result = crag_service.generate(comparison_query, confidence_threshold=0.6, verbose=False)
print(f"   Time: {crag_result['generation_time']:.2f}s")
print(f"   Correction applied: {crag_result['correction_applied']}")

# Create comparison table
comparison_data = [
    {
        'Technique': 'Standard RAG',
        'Latency (s)': f"{rag_result['generation_time']:.2f}",
        'Docs Retrieved': rag_result['num_docs'],
        'Cache Used': 'No',
        'Self-Correcting': 'No',
        'Best For': 'General queries'
    },
    {
        'Technique': 'CAG',
        'Latency (s)': f"{cag_result['generation_time']:.2f}",
        'Docs Retrieved': 'Cached' if cag_result['cache_hit'] else rag_result['num_docs'],
        'Cache Used': 'Yes' if cag_result['cache_hit'] else 'No',
        'Self-Correcting': 'No',
        'Best For': 'Frequent queries'
    },
    {
        'Technique': 'CRAG',
        'Latency (s)': f"{crag_result['generation_time']:.2f}",
        'Docs Retrieved': crag_result['docs_used'],
        'Cache Used': 'No',
        'Self-Correcting': 'Yes',
        'Best For': 'Complex/uncertain queries'
    }
]

df = pd.DataFrame(comparison_data)

print("\n" + "=" * 80)
print("COMPARISON TABLE")
print("=" * 80)
print(df.to_string(index=False))

print("\n" + "=" * 80)
print("KEY INSIGHTS")
print("=" * 80)
print("- RAG: Baseline - reliable for general queries")
print("- CAG: Fastest when cache hits - ideal for FAQs")
print("- CRAG: Most accurate - self-corrects weak evidence")
print("- HYBRID: Combine all three for production systems")
print("\n" + "=" * 80)


2026-11-06 19:33:46,304 - application.chat_service.rag_service - INFO - Processing generation request for query: 'What cardiology services are available at Nawaloka?'
2026-11-06 19:33:46,319 - application.chat_service.rag_service - INFO - Retrieved 8 evidence documents.
2026-11-06 19:33:46,343 - utils.format_utils - INFO - Formatting 8 documents into context.


RAG vs CAG vs CRAG COMPARISON

1. Standard RAG...


2026-11-06 19:33:47,152 - httpx2 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-06 19:33:49,168 - application.chat_service.rag_service - INFO - Generation complete in 2.86s | Docs: 8 | Unique URLs: 2
2026-11-06 19:33:49,168 - application.chat_service.cag_cache - INFO - Initiating cache lookup for query: 'What cardiology services are available at Nawaloka?'
2026-11-06 19:33:49,184 - application.chat_service.cag_cache - INFO - Updated History embedding matrix: 3 active entries | Dimensions: 384
2026-11-06 19:33:49,184 - application.chat_service.cag_cache - INFO - Cache MISS. Best similarity score 0.6965 was below required threshold 0.9000
2026-11-06 19:33:49,189 - application.chat_service.cag_cache - INFO - MISS [All Tiers] | No matching cached entries found for query: 'What cardiology services are available at Nawaloka?'
2026-11-06 19:33:49,189 - application.chat_service.rag_service - INFO - Processing generation request for query: 'W

   Time: 2.86s

2. Cache-Augmented Generation (CAG)...


2026-11-06 19:33:49,896 - httpx2 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-06 19:33:52,527 - application.chat_service.rag_service - INFO - Generation complete in 3.34s | Docs: 8 | Unique URLs: 2
2026-11-06 19:33:52,543 - application.chat_service.cag_cache - INFO - Cached new query to history. Generated key: '2dab3f82e31ff132e0b892db6f960676'
2026-11-06 19:33:52,543 - application.chat_service.cag_cache - INFO - Updated History embedding matrix: 4 active entries | Dimensions: 384
2026-11-06 19:33:52,543 - application.chat_service.cag_cache - INFO - Saved 4 items to History cache on disk.
2026-11-06 19:33:52,567 - utils.format_utils - INFO - Confidence Breakdown -> Keyword: 0.250 | Length: 1.000 | Diversity: 0.333 | Final: 0.492
2026-11-06 19:33:52,577 - utils.format_utils - INFO - Confidence Breakdown -> Keyword: 0.214 | Length: 1.000 | Diversity: 0.333 | Final: 0.474
2026-11-06 19:33:52,577 - utils.format_utils - INFO - Formattin

   Time: 3.34s
   Cache hit: False

3. Corrective RAG (CRAG)...


2026-11-06 19:33:53,242 - httpx2 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


   Time: 2.75s
   Correction applied: True

COMPARISON TABLE
   Technique Latency (s)  Docs Retrieved Cache Used Self-Correcting                  Best For
Standard RAG        2.86               8         No              No           General queries
         CAG        3.34               8         No              No          Frequent queries
        CRAG        2.75               8         No             Yes Complex/uncertain queries

KEY INSIGHTS
- RAG: Baseline - reliable for general queries
- CAG: Fastest when cache hits - ideal for FAQs
- CRAG: Most accurate - self-corrects weak evidence
- HYBRID: Combine all three for production systems



---
# Interactive Inference: Ask Your Own Question

Run the cell below to ask your own question and get answers from all 3 RAG systems side-by-side.


In [14]:
# Cell 14: Interactive Inference - Ask Your Own Question

# Fallback initialization if individual cells were skipped
if 'rag_service' not in dir():
    rag_service = RAGService(retriever=retriever, llm=llm, k=TOP_K_RESULTS)
if 'cag_service' not in dir():
    cache = CAGCache(cache_dir=CACHE_DIR, embedder=embeddings, similarity_threshold=0.90)
    cag_service = CAGService(rag_service=rag_service, cache=cache)
if 'crag_service' not in dir():
    from utils.config_utils import CRAG_EXPANDED_K
    crag_service = CRAGService(retriever=retriever, llm=llm, initial_k=TOP_K_RESULTS, expanded_k=CRAG_EXPANDED_K)

print("=" * 80)
print("INTERACTIVE RAG INFERENCE")
print("=" * 80)
print("\nAsk your question about Nawaloka Hospital...\n")

YOUR_QUESTION = input("Question: ")
if not YOUR_QUESTION.strip():
    YOUR_QUESTION = "What health checkup packages are offered?"

print(f"\nProcessing query: '{YOUR_QUESTION}'\n")
print("=" * 80)

results = {}

# 1. Standard RAG
print("\n1. Standard RAG")
print("-" * 80)
rag_result = rag_service.generate(YOUR_QUESTION)
results['RAG'] = {
    'answer': rag_result.get('answer', 'N/A'),
    'time': rag_result.get('generation_time', 0),
    'docs': rag_result.get('num_docs', len(rag_result.get('evidence_urls', []))),
    'urls': rag_result.get('evidence_urls', [])
}
print(f"Completed in {results['RAG']['time']:.2f}s")
print(f"Documents retrieved: {results['RAG']['docs']}")
print("\nAnswer:")
print(results['RAG']['answer'])
print("\nEvidence URLs:")
for url in results['RAG']['urls'][:3]:
    print(f"   - {url}")

# 2. Cache-Augmented Generation
print("\n" + "=" * 80)
print("\n2. Cache-Augmented Generation (CAG)")
print("-" * 80)
cag_result = cag_service.generate(YOUR_QUESTION, use_cache=True, verbose=False)
results['CAG'] = {
    'answer': cag_result.get('answer', 'N/A'),
    'time': cag_result.get('generation_time', 0),
    'docs': cag_result.get('num_docs', len(cag_result.get('evidence_urls', []))),
    'cache_hit': cag_result.get('cache_hit', False),
    'urls': cag_result.get('evidence_urls', [])
}
print(f"Completed in {results['CAG']['time']:.2f}s")
print(f"Cache hit: {results['CAG']['cache_hit']}")
print(f"Documents retrieved: {results['CAG']['docs']}")
print("\nAnswer:")
print(results['CAG']['answer'])
print("\nEvidence URLs:")
for url in results['CAG']['urls'][:3]:
    print(f"   - {url}")

# 3. Corrective RAG
print("\n" + "=" * 80)
print("\n3. Corrective RAG (CRAG)")
print("-" * 80)
crag_result = crag_service.generate(YOUR_QUESTION, confidence_threshold=0.6, verbose=False)
results['CRAG'] = {
    'answer': crag_result.get('answer', 'N/A'),
    'time': crag_result.get('generation_time', 0),
    'docs': crag_result.get('docs_used', len(crag_result.get('evidence_urls', []))),
    'confidence': crag_result.get('confidence_final', 0.0),
    'corrected': crag_result.get('correction_applied', False),
    'urls': crag_result.get('evidence_urls', [])
}
print(f"Completed in {results['CRAG']['time']:.2f}s")
print(f"Confidence: {results['CRAG']['confidence']:.2f}")
print(f"Correction applied: {results['CRAG']['corrected']}")
print(f"Documents used: {results['CRAG']['docs']}")
print("\nAnswer:")
print(results['CRAG']['answer'])
print("\nEvidence URLs:")
for url in results['CRAG']['urls'][:3]:
    print(f"   - {url}")

print("\n" + "=" * 80)
print("PERFORMANCE COMPARISON")
print("=" * 80)
print(f"\n{'System':<15} {'Time (s)':<12} {'Docs':<8} {'Special Feature':<30}")
print("-" * 80)
print(f"{'Standard RAG':<15} {results['RAG']['time']:<12.2f} {results['RAG']['docs']:<8} {'Baseline':<30}")

cag_feature = 'Cache: ' + ('HIT' if results['CAG']['cache_hit'] else 'MISS')
print(f"{'CAG':<15} {results['CAG']['time']:<12.2f} {results['CAG']['docs']:<8} {cag_feature:<30}")

crag_conf = results['CRAG']['confidence']
crag_feature = f"Confidence: {crag_conf:.2f} ({'High' if crag_conf > 0.7 else 'Needs Attention'})"
print(f"{'CRAG':<15} {results['CRAG']['time']:<12.2f} {results['CRAG']['docs']:<8} {crag_feature:<30}")

print("=" * 80)
print("RECOMMENDATION")
print("=" * 80)

fastest = min(results.items(), key=lambda x: x[1]['time'])
print(f"Fastest: {fastest[0]} ({fastest[1]['time']:.2f}s)")

if results['CAG']['cache_hit']:
    print("Best Choice: CAG (cache hit = instant response)")
elif results['CRAG']['confidence'] > 0.7:
    print("Best Choice: CRAG (high confidence + corrective capability)")
else:
    print("Best Choice: Standard RAG (reliable baseline)")

print("\n[SUCCESS] Inference complete!")
print("=" * 80)


INTERACTIVE RAG INFERENCE

Ask your question about Nawaloka Hospital...



2026-11-06 19:37:59,811 - application.chat_service.rag_service - INFO - Processing generation request for query: 'When is the cardiology availables'
2026-11-06 19:37:59,852 - application.chat_service.rag_service - INFO - Retrieved 8 evidence documents.
2026-11-06 19:37:59,880 - utils.format_utils - INFO - Formatting 8 documents into context.



Processing query: 'When is the cardiology availables'


1. Standard RAG
--------------------------------------------------------------------------------


2026-11-06 19:38:00,862 - httpx2 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-06 19:38:03,292 - application.chat_service.rag_service - INFO - Generation complete in 3.48s | Docs: 8 | Unique URLs: 2
2026-11-06 19:38:03,292 - application.chat_service.cag_cache - INFO - Initiating cache lookup for query: 'When is the cardiology availables'
2026-11-06 19:38:03,311 - application.chat_service.cag_cache - INFO - Updated History embedding matrix: 4 active entries | Dimensions: 384
2026-11-06 19:38:03,315 - application.chat_service.cag_cache - INFO - Cache MISS. Best similarity score 0.5984 was below required threshold 0.9000
2026-11-06 19:38:03,316 - application.chat_service.cag_cache - INFO - MISS [All Tiers] | No matching cached entries found for query: 'When is the cardiology availables'
2026-11-06 19:38:03,317 - application.chat_service.rag_service - INFO - Processing generation request for query: 'When is the cardiology availables'
20

Completed in 3.48s
Documents retrieved: 8

Answer:
1. **Key Facts**:
   - Nawaloka Hospital offers a range of medical specialties, including cardiology.
   - You can book an appointment with a cardiologist through their channeling service.
   - Appointments can be scheduled online via their secure booking platform.

2. **Answer**: The specific availability of cardiology appointments is not provided in the available information. To confirm availability, please contact the hospital directly or check their online booking platform [here](https://echannelling.com/?isHospitalView=true&hospital_code=H03) [URL].

3. **Contact**: For specific questions regarding cardiology availability, please call +94 11 544 4444.

Evidence URLs:
   - https://www.nawaloka.com/
   - https://www.nawaloka.com/channeling


2. Cache-Augmented Generation (CAG)
--------------------------------------------------------------------------------


2026-11-06 19:38:04,006 - httpx2 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-06 19:38:06,321 - application.chat_service.rag_service - INFO - Generation complete in 3.00s | Docs: 8 | Unique URLs: 2
2026-11-06 19:38:06,333 - application.chat_service.cag_cache - INFO - Cached new query to history. Generated key: 'd078517c84cdbdead5ff4c0e3f6a31f1'
2026-11-06 19:38:06,333 - application.chat_service.cag_cache - INFO - Updated History embedding matrix: 5 active entries | Dimensions: 384
2026-11-06 19:38:06,335 - application.chat_service.cag_cache - INFO - Saved 5 items to History cache on disk.
2026-11-06 19:38:06,351 - utils.format_utils - INFO - Confidence Breakdown -> Keyword: 0.200 | Length: 0.381 | Diversity: 0.333 | Final: 0.281
2026-11-06 19:38:06,370 - utils.format_utils - INFO - Confidence Breakdown -> Keyword: 0.150 | Length: 0.475 | Diversity: 0.333 | Final: 0.284
2026-11-06 19:38:06,371 - utils.format_utils - INFO - Formattin

Completed in 3.00s
Cache hit: False
Documents retrieved: 8

Answer:
1. **Key Facts**:
   - Nawaloka Hospital offers a channeling center for various medical specialties, including cardiology.
   - You can book an appointment with a cardiologist through their online booking platform or by contacting the channeling desk.
   - The hospital has over 600 specialist consultants across major specialties, ensuring minimal waiting times.

2. **Answer**: The specific availability of cardiology appointments is not provided in the available information. You can confirm availability by contacting the hospital directly or checking their online booking platform [here](https://echannelling.com/?isHospitalView=true&hospital_code=H03) [URL].

3. **Contact**: For specific questions regarding cardiology availability, please call +94 11 544 4444.

Evidence URLs:
   - https://www.nawaloka.com/
   - https://www.nawaloka.com/channeling


3. Corrective RAG (CRAG)
------------------------------------------------

2026-11-06 19:38:07,025 - httpx2 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


Completed in 3.10s
Confidence: 0.28
Correction applied: True
Documents used: 8

Answer:
1. **Key Facts**:
   - Nawaloka Hospital offers a channeling center for various medical specialties, including cardiology.
   - You can book an appointment with a cardiologist through their online booking platform or by contacting the channeling desk.
   - The hospital has over 600 specialist consultants across major specialties, ensuring minimal waiting times.

2. **Answer**: The specific availability of cardiology appointments is not provided in the available information. To confirm availability, please contact the hospital directly or check their online booking platform [here](https://echannelling.com/?isHospitalView=true&hospital_code=H03) [URL].

3. **Contact**: For specific questions regarding cardiology availability, please call +94 11 544 4444.

Evidence URLs:
   - https://www.nawaloka.com/
   - https://www.nawaloka.com/channeling

PERFORMANCE COMPARISON

System          Time (s)     Docs   

---
# Summary

All three RAG techniques implemented using service layer:
- **RAGService**: Standard RAG with modern LCEL
- **CAGService**: Cache-augmented generation with semantic matching
- **CRAGService**: Corrective RAG with confidence scoring

**CAG Semantic Caching**:
- Catches paraphrases: "What are visiting hours?" matches "Tell me the visiting hours"
- Two-tier: Static FAQs (never expire) + Dynamic History (24h TTL)
- Lightweight: Only new queries need embedding, cached ones use stored embeddings
- Fast lookup: Cosine similarity is just a dot product (~1ms for 1000 entries)

**Benefits of service-based architecture**:
- Reusable across notebooks and production code
- Easily testable
- Well-documented
- Maintainable (logic in one place)

**OpenRouter Multi-Provider Support**:
- One API key -> access to OpenAI, Anthropic, Google, Meta, DeepSeek
- Configure models in `config/model.yaml`
- Switch providers without code changes
